# Model Reconciliation: Published vs ModelSEED

This notebook compares the published ADP1 metabolic model against the ModelSEED-reconstructed model,
then merges unique published reactions into the ModelSEED model after ATP safety filtering.

**Workflow:**
1. Load both models (published from local JSON, ModelSEED from KBase)
2. Run pFBA and FVA (50% optimal) on both models in pyruvate minimal media
3. Build a comprehensive comparison dataframe of all reactions across both models
4. Identify published-only reactions with genes absent from the ModelSEED model
5. Save the ModelSEED model locally as JSON
6. Merge published reactions into the ModelSEED model; run ATP safe expansion to filter ATP-breaking reactions
7. Run pFBA and FVA on the merged model in pyruvate media
8. Save the final merged model
9. Generate an Escher map with pyruvate fluxes and reaction class badges (source × FVA class)

## Cell 1: Load Both Models

- **Published model**: loaded from `models/FullyTranslatedPublishedModel.json` (already translated to ModelSEED namespace)
- **ModelSEED model**: loaded from KBase workspace via `util.get_model()`
- Both models are saved to datacache for cell independence

In [ ]:
%run util.py

# Load published model from local JSON
pub_mdlutl = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")
print(f"Published model: {len(pub_mdlutl.model.reactions)} reactions, "
      f"{len(pub_mdlutl.model.metabolites)} metabolites, "
      f"{len(pub_mdlutl.model.genes)} genes")
print(f"  Biomass reaction: GROWTH_DASH_RXN")

# Load ModelSEED model from KBase
ms_mdlutl = util.get_model("179225/Abaylyi_ADP1_RASTMS2_OMEGGA_Abaylyi_ADP1_RAST.mdlMS2_OMEGGA_iAbaylyi_Carbon_Succinic.gf")
# Remove mRNA_ genes
gene_remove_list = [g for g in ms_mdlutl.model.genes if str(g.id).startswith("mRNA_")]
for gene in gene_remove_list:
    ms_mdlutl.model.genes.remove(gene)
print(f"ModelSEED model: {len(ms_mdlutl.model.reactions)} reactions, "
      f"{len(ms_mdlutl.model.metabolites)} metabolites, "
      f"{len(ms_mdlutl.model.genes)} genes")
print(f"  Biomass reaction: bio1")

# Save model summaries to datacache
model_info = {
    "published": {
        "reactions": len(pub_mdlutl.model.reactions),
        "metabolites": len(pub_mdlutl.model.metabolites),
        "genes": len(pub_mdlutl.model.genes),
        "biomass_rxn": "GROWTH_DASH_RXN"
    },
    "modelseed": {
        "reactions": len(ms_mdlutl.model.reactions),
        "metabolites": len(ms_mdlutl.model.metabolites),
        "genes": len(ms_mdlutl.model.genes),
        "biomass_rxn": "bio1"
    }
}
util.save("ModelReconciliation/model_info", model_info)
print("\nModel info saved to datacache.")

## Cell 2: Run pFBA and FVA on Both Models in Pyruvate Minimal Media

- Set pyruvate minimal media on both models
- Run pFBA to get optimal flux distribution
- Run FVA at 50% optimal growth to classify reaction flux flexibility
- Results cached separately for each model in `ModelReconciliation/` subdirectory

In [1]:
%run util.py

# Load published model
pub_mdlutl = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")

# Load ModelSEED model
ms_mdlutl = util.get_model("179225/Abaylyi_ADP1_RASTMS2_OMEGGA_Abaylyi_ADP1_RAST.mdlMS2_OMEGGA_iAbaylyi_Carbon_Succinic.gf")
gene_remove_list = [g for g in ms_mdlutl.model.genes if str(g.id).startswith("mRNA_")]
for gene in gene_remove_list:
    ms_mdlutl.model.genes.remove(gene)

# --- Published model: pFBA + FVA ---
print("=== Published Model ===")
pub_pfba = util.run_fba(pub_mdlutl, media="KBaseMedia/Carbon-Pyruvic-Acid", 
                         objective="MAX{GROWTH_DASH_RXN}", run_pfba=True)
print(f"pFBA growth rate: {pub_pfba.objective_value:.6f}")

pub_fva = util.run_fva(pub_mdlutl, media="KBaseMedia/Carbon-Pyruvic-Acid",
                        objective="MAX{GROWTH_DASH_RXN}", fraction_of_optimum=0.5)
print(f"FVA completed for {len(pub_fva)} reactions at 50% optimum")

# Package published results
pub_results = {
    "growth_rate": pub_pfba.objective_value,
    "fluxes": {rxn_id: float(val) for rxn_id, val in pub_pfba.fluxes.items()},
    "fva": pub_fva
}
util.save("ModelReconciliation/published_pyr_results", pub_results)
print("Published results saved.\n")

# --- ModelSEED model: pFBA + FVA ---
print("=== ModelSEED Model ===")
ms_pfba = util.run_fba(ms_mdlutl, media="KBaseMedia/Carbon-Pyruvic-Acid",
                        objective="MAX{bio1}", run_pfba=True)
print(f"pFBA growth rate: {ms_pfba.objective_value:.6f}")

ms_fva = util.run_fva(ms_mdlutl, media="KBaseMedia/Carbon-Pyruvic-Acid",
                       objective="MAX{bio1}", fraction_of_optimum=0.5)
print(f"FVA completed for {len(ms_fva)} reactions at 50% optimum")

# Package modelseed results
ms_results = {
    "growth_rate": ms_pfba.objective_value,
    "fluxes": {rxn_id: float(val) for rxn_id, val in ms_pfba.fluxes.items()},
    "fva": ms_fva
}
util.save("ModelReconciliation/modelseed_pyr_results", ms_results)
print("ModelSEED results saved.")

/Users/chenry/Dropbox/Projects/KBUtilLib/src
modelseedpy 0.4.2


2026-02-02 22:40:39,487 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-02 22:40:39,488 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-02 22:40:39,488 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token


loading biochemistry database from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


2026-02-02 22:40:44,237 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase
2026-02-02 22:40:44,635 - __main__.NotebookUtil - WARNING - BLAST tools not found. Install NCBI BLAST+ to use BLAST functionality. On Ubuntu/Debian: sudo apt-get install ncbi-blast+, On MacOS: brew install blast
2026-02-02 22:40:44,636 - __main__.NotebookUtil - INFO - Notebook name: ModelReconciliation
2026-02-02 22:40:44,637 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-02 22:40:44,637 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/


cobrakbase 0.4.0
/Users/chenry/.npm-global/bin/claude


...

=== Published Model ===


INFO:modelseedpy.core.msmodelutl:cpd00063 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd10516 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00205 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00254 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00099 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00058 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00030 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00034 not found in model!


pFBA growth rate: 0.446864


INFO:modelseedpy.core.msmodelutl:cpd00063 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd10516 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00205 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00254 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00099 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00058 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00030 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00034 not found in model!


FVA completed for 990 reactions at 50% optimum
Published results saved.

=== ModelSEED Model ===
pFBA growth rate: 0.209254
FVA completed for 1417 reactions at 50% optimum
ModelSEED results saved.


## Cell 3: Build Comprehensive Comparison Dataframe

Builds a dataframe listing all reactions across both models with columns:
- **ID**: Reaction ID (standardized for exchanges as `EX_cpd<ID>`)
- **published_pyr_flux**: Flux value and FVA class from published model
- **modelseed_pyr_flux**: Flux value and FVA class from ModelSEED model
- **membership**: `both`, `published`, or `modelseed`
- **extra_published_genes**: Genes in published model not in ModelSEED (with MS rxn if mapped)
- **extra_modelseed_genes**: Genes in ModelSEED model not in published (with pub rxn if mapped)
- **directionality**: Bound directionality comparison (published/MS)
- **equation**: Reaction equation with metabolite names
- **stoichiometry_differences**: Differences in stoichiometry/compartments between models

In [1]:
%run util.py

# Load cached results
pub_results = util.load("ModelReconciliation/published_pyr_results")
ms_results = util.load("ModelReconciliation/modelseed_pyr_results")

# Reload models for structure inspection
pub_mdlutl = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")
ms_mdlutl = util.get_model("179225/Abaylyi_ADP1_RASTMS2_OMEGGA_Abaylyi_ADP1_RAST.mdlMS2_OMEGGA_iAbaylyi_Carbon_Succinic.gf")
gene_remove_list = [g for g in ms_mdlutl.model.genes if str(g.id).startswith("mRNA_")]
for gene in gene_remove_list:
    ms_mdlutl.model.genes.remove(gene)

pub_model = pub_mdlutl.model
ms_model = ms_mdlutl.model

# Build exchange ID maps: standardized_id -> original_rxn_id
pub_ex_map = util.get_exchange_map(pub_model)
ms_ex_map = util.get_exchange_map(ms_model)

# Build reverse maps: original_rxn_id -> standardized_id
pub_ex_rev = {v: k for k, v in pub_ex_map.items()}
ms_ex_rev = {v: k for k, v in ms_ex_map.items()}

# Build gene-to-reaction maps for cross-referencing extra genes
pub_gene_rxn_map = util.build_gene_reaction_map(pub_model)
ms_gene_rxn_map = util.build_gene_reaction_map(ms_model)

# Collect all reaction IDs using standardized exchange names
# For each reaction, determine its "canonical ID" for comparison
def canonical_id(rxn, ex_rev_map):
    """Return standardized ID for exchanges, original ID otherwise."""
    if rxn.id in ex_rev_map:
        return ex_rev_map[rxn.id]
    return rxn.id

# Map canonical_id -> original reaction object for each model
pub_rxn_by_cid = {}
for rxn in pub_model.reactions:
    cid = canonical_id(rxn, pub_ex_rev)
    pub_rxn_by_cid[cid] = rxn

ms_rxn_by_cid = {}
for rxn in ms_model.reactions:
    cid = canonical_id(rxn, ms_ex_rev)
    ms_rxn_by_cid[cid] = rxn

# All canonical IDs across both models
all_cids = sorted(set(pub_rxn_by_cid.keys()) | set(ms_rxn_by_cid.keys()))
print(f"Total unique reaction IDs (canonical): {len(all_cids)}")
print(f"  Published only: {len(set(pub_rxn_by_cid.keys()) - set(ms_rxn_by_cid.keys()))}")
print(f"  ModelSEED only: {len(set(ms_rxn_by_cid.keys()) - set(pub_rxn_by_cid.keys()))}")
print(f"  In both: {len(set(pub_rxn_by_cid.keys()) & set(ms_rxn_by_cid.keys()))}")

# Build the comparison rows
rows = []
pub_fluxes = pub_results["fluxes"]
ms_fluxes = ms_results["fluxes"]
pub_fva = pub_results["fva"]
ms_fva = ms_results["fva"]

# Sets of genes in each model for cross-referencing
pub_gene_set = set(str(g) for g in pub_model.genes if not str(g).startswith("mRNA_"))
ms_gene_set = set(str(g) for g in ms_model.genes if not str(g).startswith("mRNA_"))

for cid in all_cids:
    pub_rxn = pub_rxn_by_cid.get(cid)
    ms_rxn = ms_rxn_by_cid.get(cid)

    # Membership
    if pub_rxn and ms_rxn:
        membership = "both"
    elif pub_rxn:
        membership = "published"
    else:
        membership = "modelseed"

    # Published flux + FVA class
    if pub_rxn:
        pub_flux_val = pub_fluxes.get(pub_rxn.id, 0.0)
        pub_fva_entry = pub_fva.get(pub_rxn.id, {"MIN": 0, "MAX": 0})
        pub_class = util.classify_fva_flux(pub_fva_entry, pub_flux_val)
        pub_flux_str = f"{pub_flux_val:.6g} ({pub_class})"
    else:
        pub_flux_str = ""

    # ModelSEED flux + FVA class
    if ms_rxn:
        ms_flux_val = ms_fluxes.get(ms_rxn.id, 0.0)
        ms_fva_entry = ms_fva.get(ms_rxn.id, {"MIN": 0, "MAX": 0})
        ms_class = util.classify_fva_flux(ms_fva_entry, ms_flux_val)
        ms_flux_str = f"{ms_flux_val:.6g} ({ms_class})"
    else:
        ms_flux_str = ""

    # Genes
    pub_genes = set()
    ms_genes = set()
    if pub_rxn:
        pub_genes = set(str(g) for g in pub_rxn.genes if not str(g).startswith("mRNA_"))
    if ms_rxn:
        ms_genes = set(str(g) for g in ms_rxn.genes if not str(g).startswith("mRNA_"))

    # Shared genes: in both models for this reaction
    shared_genes = "; ".join(sorted(pub_genes & ms_genes))

    # Extra published genes: in published but not in ModelSEED for this reaction
    extra_pub_genes_list = []
    for g in sorted(pub_genes - ms_genes):
        # Check if this gene maps to any reaction in the ModelSEED model
        ms_rxns_for_gene = ms_gene_rxn_map.get(g, [])
        if ms_rxns_for_gene:
            extra_pub_genes_list.append(f"{g} ({','.join(ms_rxns_for_gene)})")
        else:
            extra_pub_genes_list.append(g)
    extra_pub_genes = "; ".join(extra_pub_genes_list)

    # Extra modelseed genes: in ModelSEED but not in published for this reaction
    extra_ms_genes_list = []
    for g in sorted(ms_genes - pub_genes):
        # Check if this gene maps to any reaction in the published model
        pub_rxns_for_gene = pub_gene_rxn_map.get(g, [])
        if pub_rxns_for_gene:
            extra_ms_genes_list.append(f"{g} ({','.join(pub_rxns_for_gene)})")
        else:
            extra_ms_genes_list.append(g)
    extra_ms_genes = "; ".join(extra_ms_genes_list)

    # Directionality
    if pub_rxn and ms_rxn:
        pub_dir = util.get_reaction_directionality(pub_rxn)
        ms_dir = util.get_reaction_directionality(ms_rxn)
        directionality = f"{pub_dir}/{ms_dir}"
    elif pub_rxn:
        directionality = util.get_reaction_directionality(pub_rxn)
    else:
        directionality = util.get_reaction_directionality(ms_rxn)

    # Equation with metabolite names (prefer published if available)
    ref_rxn = pub_rxn if pub_rxn else ms_rxn
    equation = util.reaction_equation_with_names(ref_rxn)

    # Stoichiometry/compartment differences
    if pub_rxn and ms_rxn:
        stoich_diffs = util.compare_reaction_stoichiometry(pub_rxn, ms_rxn)
        stoich_diff_str = "; ".join(stoich_diffs) if stoich_diffs else ""
    else:
        stoich_diff_str = ""

    rows.append({
        "ID": cid,
        "published_pyr_flux": pub_flux_str,
        "modelseed_pyr_flux": ms_flux_str,
        "membership": membership,
        "shared_genes": shared_genes,
        "extra_published_genes": extra_pub_genes,
        "extra_modelseed_genes": extra_ms_genes,
        "directionality": directionality,
        "equation": equation,
        "stoichiometry_differences": stoich_diff_str
    })

# Create DataFrame
df = pd.DataFrame(rows)
print(f"\nComparison dataframe: {len(df)} rows")
print(f"Membership counts:")
print(df["membership"].value_counts().to_string())

# Save to datacache and nboutput
util.save("ModelReconciliation/comparison_dataframe", df.to_dict(orient="records"))
df.to_csv(f"{util.output_dir}/model_reconciliation.tsv", sep="\t", index=False)
print(f"\nSaved to datacache and nboutput/model_reconciliation.tsv")

# Display
display(df)

/Users/chenry/Dropbox/Projects/KBUtilLib/src
modelseedpy 0.4.2


2026-02-03 12:44:44,922 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-03 12:44:44,923 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-03 12:44:44,924 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token


loading biochemistry database from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


2026-02-03 12:44:49,632 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase
2026-02-03 12:44:50,032 - __main__.NotebookUtil - WARNING - BLAST tools not found. Install NCBI BLAST+ to use BLAST functionality. On Ubuntu/Debian: sudo apt-get install ncbi-blast+, On MacOS: brew install blast
2026-02-03 12:44:50,034 - __main__.NotebookUtil - INFO - Notebook name: ModelReconciliation
2026-02-03 12:44:50,034 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-03 12:44:50,035 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/


cobrakbase 0.4.0
/Users/chenry/.npm-global/bin/claude


2026-02-03 12:44:51,859 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: claude-code


Total unique reaction IDs (canonical): 1804
  Published only: 387
  ModelSEED only: 814
  In both: 603

Comparison dataframe: 1804 rows
Membership counts:
membership
modelseed    814
both         603
published    387

Saved to datacache and nboutput/model_reconciliation.tsv


,ID,published_pyr_flux,modelseed_pyr_flux,membership,shared_genes,extra_published_genes,extra_modelseed_genes,directionality,equation,stoichiometry_differences
0,1_DASH_ACYLGLYCEROL_DASH_3_DASH_P_DASH_ACYLTRA...,0.025184 (essential_fwd),,published,,"ACIAD_RS14655 (rxn10209_c0,rxn10205_c0,rxn0855...",,reversible,generic acyl-glycerol-3-phosphate for phosphol...,
1,2_DASH_METHYLCITRATE_DASH_DEHYDRATASE_DASH_RXN,0 (optionally_active_zero_rev),,published,,ACIAD_RS12480,,reversible,2-methylcitrate <=> h2o + 2-methyl-cis-aconitate,
2,3_DASH_DEHYDROQUINATE_DASH_DEHYDRATASE_DASH_RXN,0 (variable_zero),,published,,ACIAD_RS08015 (rxn02213_c0),,reversible,3-dehydroquinate <=> h2o + 3-dehydro-shikimate,
3,ACETOOHBUTREDUCTOISOM_DASH_RXN,0.14315 (essential_fwd),,published,,"ACIAD_RS14005 (rxn03436_c0,rxn03435_c0,rxn0218...",,reversible,nadph + 2-aceto-2-hydroxy-butyrate <=> nadp + ...,
4,C14_COLON_0_DASH_3HYDROXYACP_DASH_DEHYDRATASE_...,0 (variable_zero),,published,,"ACIAD_RS06370 (rxn05334_c0,rxn05386_c0,rxn0539...",,reversible,3-hydroxytetradecanoyl-acp <=> h2o + trans-tet...,
...,...,...,...,...,...,...,...,...,...,...
1799,rxn47831_c0,0.0572601 (essential_fwd),,published,,ACIAD_RS13745 (rxn05296_c0); ACIAD_RS13750 (rx...,,reversible,atp + l-phenylalanine + phenylalanine trna <=>...,
1800,rxn47838_c0,0.11452 (essential_fwd),,published,,ACIAD_RS02785 (rxn05296_c0),,reversible,atp + l-aspartate + aspartate trna <=> pyropho...,
1801,rxn48235_c0,0.14315 (essential_fwd),,published,,ACIAD_RS00120 (rxn05296_c0),,reversible,atp + l-iso-leucine + isoleucine trna <=> pyro...,
1802,rxn48506_c0,0.372191 (essential_fwd),,published,,ACIAD_RS05770 (rxn05296_c0),,reversible,atp + l-alanine + alanine trna <=> pyrophospha...,


## Cell 4: Identify Published-Only Reactions with Non-Overlapping Genes

Produces a list of reactions that are:
1. **Only in the published model** (not in ModelSEED model)
2. **Associated with genes that do NOT appear anywhere in the ModelSEED model**
3. **Not exchange reactions** (single-metabolite reactions are excluded)

These represent reactions from the published literature model whose gene associations
are entirely absent from the ModelSEED reconstruction — candidates for merging into
the ModelSEED model to improve coverage.

In [2]:
%run util.py

# Load comparison data
comparison = util.load("ModelReconciliation/comparison_dataframe")

# Reload models for gene set extraction
pub_mdlutl = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")
ms_mdlutl = util.get_model("179225/Abaylyi_ADP1_RASTMS2_OMEGGA_Abaylyi_ADP1_RAST.mdlMS2_OMEGGA_iAbaylyi_Carbon_Succinic.gf")
gene_remove_list = [g for g in ms_mdlutl.model.genes if str(g.id).startswith("mRNA_")]
for gene in gene_remove_list:
    ms_mdlutl.model.genes.remove(gene)

pub_model = pub_mdlutl.model
ms_model = ms_mdlutl.model

# Build the set of ALL genes in the ModelSEED model
ms_all_genes = set(str(g) for g in ms_model.genes)

# Identify published-only reactions with genes NOT in ModelSEED model (exclude exchanges)
published_only_unique_gene_rxns = []
for rxn in pub_model.reactions:
    # Skip exchange reactions (single metabolite)
    if len(rxn.metabolites) == 1:
        continue
    # Skip biomass
    if rxn.id.startswith("bio") or "GROWTH" in rxn.id or "BIOMASS" in rxn.id:
        continue
    # Check if this reaction exists in the ModelSEED model (by canonical ID)
    std_id = util.standardize_exchange_id(rxn)
    ms_has_rxn = std_id in [r.id for r in ms_model.reactions] or rxn.id in [r.id for r in ms_model.reactions]
    if ms_has_rxn:
        continue
    # Get genes for this reaction (excluding mRNA_ prefix)
    rxn_genes = set(str(g) for g in rxn.genes if not str(g).startswith("mRNA_"))
    if not rxn_genes:
        continue  # Skip spontaneous reactions (no genes)
    # Check if ANY of this reaction's genes appear in the ModelSEED model
    overlapping_genes = rxn_genes & ms_all_genes
    if len(overlapping_genes) == 0:
        # None of this reaction's genes are in the MS model
        published_only_unique_gene_rxns.append({
            "rxn_id": rxn.id,
            "genes": sorted(rxn_genes),
            "gene_rule": rxn.gene_reaction_rule,
            "equation": util.reaction_equation_with_names(rxn),
            "lower_bound": rxn.lower_bound,
            "upper_bound": rxn.upper_bound,
        })

print(f"Published-only reactions with genes NOT in ModelSEED model: {len(published_only_unique_gene_rxns)}")
print(f"(Excludes exchange reactions and spontaneous reactions)\n")

# Display as DataFrame
df_unique = pd.DataFrame(published_only_unique_gene_rxns)
if len(df_unique) > 0:
    df_unique["genes_str"] = df_unique["genes"].apply(lambda x: "; ".join(x))
    display(df_unique[["rxn_id", "genes_str", "gene_rule", "equation", "lower_bound", "upper_bound"]])

# Save to datacache
util.save("ModelReconciliation/published_only_unique_gene_rxns", published_only_unique_gene_rxns)
print(f"\nSaved to datacache/ModelReconciliation/published_only_unique_gene_rxns.json")

2026-02-03 12:44:58,273 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-03 12:44:58,273 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-03 12:44:58,274 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-02-03 12:44:58,275 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


/Users/chenry/Dropbox/Projects/KBUtilLib/src


2026-02-03 12:44:58,653 - __main__.NotebookUtil - WARNING - BLAST tools not found. Install NCBI BLAST+ to use BLAST functionality. On Ubuntu/Debian: sudo apt-get install ncbi-blast+, On MacOS: brew install blast
2026-02-03 12:44:58,655 - __main__.NotebookUtil - INFO - Notebook name: ModelReconciliation
2026-02-03 12:44:58,655 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-03 12:44:58,656 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/


/Users/chenry/.npm-global/bin/claude


2026-02-03 12:45:00,140 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: claude-code


Published-only reactions with genes NOT in ModelSEED model: 70
(Excludes exchange reactions and spontaneous reactions)



,rxn_id,genes_str,gene_rule,equation,lower_bound,upper_bound
0,rxn12232_c0,ACIAD_RS08440,ACIAD_RS08440,o2 + 1-octanal + fmnh2 <=> h2o + fmn + octanoate,-1000.0,1000.0
1,rxn12393_c0,ACIAD_RS03860; ACIAD_RS06595; ACIAD_RS12605; A...,ACIAD_RS03860 or ACIAD_RS06595 or ACIAD_RS1260...,h+ + hexanoate <=> h+ + hexanoate,-1000.0,1000.0
2,rxn12217_c0,SPONT,SPONT,"formaldehyde + thf <=> h2o + 5,10-methylene-thf",-1000.0,1000.0
3,rxn12356_c0,ACIAD_RS15310,ACIAD_RS15310,nadph + cis-octadec-9-enoyl-coa <=> nadp + coe...,-1000.0,1000.0
4,rxn12348_c0,ACIAD_RS05140; ACIAD_RS14975,ACIAD_RS14975 or ACIAD_RS05140,"h2o + triacylglycerol <=> 1,2-diacylglycerol +...",-1000.0,1000.0
...,...,...,...,...,...,...
65,rxn00595_c0,ACIAD_RS12135; ACIAD_RS12140; ACIAD_RS12145,ACIAD_RS12135 and ACIAD_RS12140 and ACIAD_RS12145,nadph + o2 + 2-aminobenzoate <=> nadp + co2 + ...,-1000.0,1000.0
66,rxn12231_c0,ACIAD_RS07415; ACIAD_RS07420,ACIAD_RS07415 and ACIAD_RS07420,h2o + propane nitrile <=> propane amide,-1000.0,1000.0
67,rxn12387_c0,ACIAD_RS03860; ACIAD_RS06595; ACIAD_RS12605; A...,ACIAD_RS03860 or ACIAD_RS06595 or ACIAD_RS1260...,h+ + tetradecanoate <=> h+ + tetradecanoate,-1000.0,1000.0
68,rxn00779_c0,ACIAD_RS02485,ACIAD_RS02485,h2o + nadp + glyceraldehyde-3-phosphate <=> na...,-1000.0,1000.0



Saved to datacache/ModelReconciliation/published_only_unique_gene_rxns.json


## Cell 5: Save ModelSEED Model Locally as JSON

Pull the ModelSEED model from KBase and save it to `models/ModelSEED_ADP1.json`
so subsequent cells can load it locally without KBase network access.

In [3]:
%run util.py

# Load ModelSEED model from KBase
ms_mdlutl = util.get_model("179225/Abaylyi_ADP1_RASTMS2_OMEGGA_Abaylyi_ADP1_RAST.mdlMS2_OMEGGA_iAbaylyi_Carbon_Succinic.gf")
# Remove mRNA_ genes
gene_remove_list = [g for g in ms_mdlutl.model.genes if str(g.id).startswith("mRNA_")]
for gene in gene_remove_list:
    ms_mdlutl.model.genes.remove(gene)

# Save as COBRApy JSON
output_path = "models/ModelSEED_ADP1.json"
cobra.io.save_json_model(ms_mdlutl.model, output_path)
print(f"ModelSEED model saved to {output_path}")
print(f"  Reactions: {len(ms_mdlutl.model.reactions)}")
print(f"  Metabolites: {len(ms_mdlutl.model.metabolites)}")
print(f"  Genes: {len(ms_mdlutl.model.genes)}")

2026-02-03 12:45:47,801 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-03 12:45:47,801 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-03 12:45:47,802 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-02-03 12:45:47,803 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


/Users/chenry/Dropbox/Projects/KBUtilLib/src


2026-02-03 12:45:48,192 - __main__.NotebookUtil - WARNING - BLAST tools not found. Install NCBI BLAST+ to use BLAST functionality. On Ubuntu/Debian: sudo apt-get install ncbi-blast+, On MacOS: brew install blast
2026-02-03 12:45:48,194 - __main__.NotebookUtil - INFO - Notebook name: ModelReconciliation
2026-02-03 12:45:48,195 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-03 12:45:48,195 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/


/Users/chenry/.npm-global/bin/claude


2026-02-03 12:45:49,873 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: claude-code


ModelSEED model saved to models/ModelSEED_ADP1.json
  Reactions: 1417
  Metabolites: 1270
  Genes: 987


## Cell 6: Merge Published Reactions and Run ATP Safe Expansion

Takes the published-only reactions with non-overlapping genes (from Cell 4) and adds them
to the ModelSEED model. Then uses `MSATPCorrection` to:
1. Build ATP production tests on the base ModelSEED model
2. Run `reaction_expansion_test` to identify which added reactions break ATP production
3. Remove ATP-breaking reactions

The result is a merged model that includes published reactions that don't disrupt ATP energetics.

In [7]:
%run util.py

# Load ModelSEED model from local JSON (saved in Cell 5)
ms_mdlutl = MSModelUtil.from_cobrapy("models/ModelSEED_ADP1.json")
ms_model = ms_mdlutl.model
print(f"Base ModelSEED model: {len(ms_model.reactions)} rxns, {len(ms_model.metabolites)} mets, {len(ms_model.genes)} genes")

# Load published model (source of reactions to add)
pub_mdlutl = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")
pub_model = pub_mdlutl.model

# Load published-only unique gene reactions list from Cell 4
unique_rxn_list = util.load("ModelReconciliation/published_only_unique_gene_rxns")
unique_rxn_ids = set(r["rxn_id"] for r in unique_rxn_list)
print(f"Total candidate reactions from Cell 4: {len(unique_rxn_ids)}")

# Exclude single-compound diffusion reactions (e.g. formate <=> formate)
diffusion_rxn_ids = set()
for rxn_id in list(unique_rxn_ids):
    pub_rxn = pub_model.reactions.get_by_id(rxn_id)
    if util.is_diffusion_reaction(pub_rxn):
        diffusion_rxn_ids.add(rxn_id)
        unique_rxn_ids.discard(rxn_id)

print(f"Single-compound diffusion reactions excluded: {len(diffusion_rxn_ids)}")
for rid in sorted(diffusion_rxn_ids):
    pub_rxn = pub_model.reactions.get_by_id(rid)
    eq = util.reaction_equation_with_names(pub_rxn)
    print(f"  {rid}: {eq}")

# Manual exclusions: reactions known to create flux loops in the merged model
manual_exclusions = {"rxn08856_c0","rxn00779_c0","rxn12504_c0","rxn03630_c0"}  # creates loop with rxn08854_c0
for rxn_id in manual_exclusions:
    if rxn_id in unique_rxn_ids:
        unique_rxn_ids.discard(rxn_id)
        print(f"Manually excluded: {rxn_id} (flux loop with rxn08854_c0)")

print(f"Candidate reactions after filtering: {len(unique_rxn_ids)}")

# --- Step 1: Add published reactions + metabolites + exchange reactions to the MS model ---
reactions_to_add = []
exchange_rxns_to_add = []
for rxn_id in unique_rxn_ids:
    pub_rxn = pub_model.reactions.get_by_id(rxn_id)
    # Build a new reaction for the MS model
    new_rxn = Reaction(pub_rxn.id)
    new_rxn.name = pub_rxn.name
    new_rxn.lower_bound = pub_rxn.lower_bound
    new_rxn.upper_bound = pub_rxn.upper_bound
    new_rxn.gene_reaction_rule = pub_rxn.gene_reaction_rule
    
    # Map metabolites, adding missing ones with normalized compartments
    met_dict = {}
    for met, coeff in pub_rxn.metabolites.items():
        if met.id in ms_model.metabolites:
            met_dict[ms_model.metabolites.get_by_id(met.id)] = coeff
        else:
            new_met = Metabolite(met.id, name=met.name,
                                 compartment=util.normalize_compartment(met.compartment),
                                 formula=met.formula, charge=met.charge)
            met_dict[new_met] = coeff
    new_rxn.add_metabolites(met_dict)
    reactions_to_add.append(new_rxn)

ms_model.add_reactions(reactions_to_add)
print(f"\nAdded {len(reactions_to_add)} reactions to model")

# Add exchange reactions for any new extracellular metabolites that lack exchanges
for met in ms_model.metabolites:
    if met.compartment == "e0":
        has_exchange = any(
            len(rxn.metabolites) == 1 and met in rxn.metabolites
            for rxn in met.reactions
        )
        if not has_exchange:
            ex_rxn = Reaction(f"EX_{met.id}")
            ex_rxn.name = f"Exchange for {met.name}"
            ex_rxn.lower_bound = -1000
            ex_rxn.upper_bound = 1000
            ex_rxn.add_metabolites({met: -1})
            exchange_rxns_to_add.append(ex_rxn)

if exchange_rxns_to_add:
    ms_model.add_reactions(exchange_rxns_to_add)
    print(f"Added {len(exchange_rxns_to_add)} exchange reactions for new extracellular metabolites")

print(f"Model after additions: {len(ms_model.reactions)} rxns, {len(ms_model.metabolites)} mets, {len(ms_model.genes)} genes")

# --- Step 2: Run ATP safe expansion ---
# Build the reaction list for expansion testing (only the newly added reactions)
new_rxn_expansion_list = []
for rxn_id in unique_rxn_ids:
    if rxn_id in ms_model.reactions:
        rxn = ms_model.reactions.get_by_id(rxn_id)
        if rxn.upper_bound > 0:
            new_rxn_expansion_list.append([rxn, ">"])
        if rxn.lower_bound < 0:
            new_rxn_expansion_list.append([rxn, "<"])
        # Temporarily disable the reaction for expansion testing
        rxn.lower_bound = 0
        rxn.upper_bound = 0

print(f"\nReaction directions to test: {len(new_rxn_expansion_list)}")

# Create MSATPCorrection to build ATP tests
atp_correction = MSATPCorrection(ms_mdlutl, load_default_medias=True)

# Evaluate which media work for ATP and determine growth media
print("Evaluating ATP production on default media...")
atp_correction.evaluate_growth_media(no_gapfilling=True)
atp_correction.determine_growth_media()
atp_correction.restore_noncore_reactions()

# Build the tests
tests = atp_correction.build_tests()
print(f"Built {len(tests)} ATP tests from {len(atp_correction.selected_media)} selected media")

# Run expansion test on the new reactions
print("Running ATP safe expansion test on added reactions...")
filtered_rxns = ms_mdlutl.reaction_expansion_test(
    new_rxn_expansion_list, tests, attribute_label="published_rxn_atp_filter"
)

# Process results
if filtered_rxns is None:
    print("WARNING: No valid solution found - expansion test could not complete")
    filtered_rxn_ids = set()
else:
    filtered_rxn_ids = set()
    for item in filtered_rxns:
        rxn = item[0]
        direction = item[1]
        filtered_rxn_ids.add(rxn.id)
        if direction == ">":
            rxn.upper_bound = 0
        else:
            rxn.lower_bound = 0
        # Remove reaction entirely if both directions are blocked
        if rxn.lower_bound == 0 and rxn.upper_bound == 0:
            ms_model.remove_reactions([rxn])

    print(f"\nATP expansion filter results:")
    print(f"  Reactions/directions tested: {len(new_rxn_expansion_list)}")
    print(f"  Filtered (ATP-breaking): {len(filtered_rxns)}")
    print(f"  Unique reactions removed or constrained: {len(filtered_rxn_ids)}")

# Count how many of the original added reactions survived
surviving_rxn_ids = unique_rxn_ids - filtered_rxn_ids
removed_rxn_ids = unique_rxn_ids & filtered_rxn_ids
# Some may have been partially constrained (one direction blocked) but still in model
still_in_model = set(r["rxn_id"] for r in unique_rxn_list if r["rxn_id"] in [rxn.id for rxn in ms_model.reactions])

print(f"\nFinal merged model: {len(ms_model.reactions)} rxns, {len(ms_model.metabolites)} mets, {len(ms_model.genes)} genes")
print(f"  Published reactions retained: {len(still_in_model)} / {len(unique_rxn_ids)}")

# Save results to datacache
merge_results = {
    "reactions_added": len(unique_rxn_ids),
    "reactions_filtered_atp": len(filtered_rxn_ids) if filtered_rxns else 0,
    "reactions_retained": len(still_in_model),
    "filtered_rxn_ids": sorted(filtered_rxn_ids),
    "retained_rxn_ids": sorted(still_in_model),
    "diffusion_rxn_ids": sorted(diffusion_rxn_ids),
    "manual_exclusion_ids": sorted(manual_exclusions),
    "final_model_reactions": len(ms_model.reactions),
    "final_model_metabolites": len(ms_model.metabolites),
    "final_model_genes": len(ms_model.genes),
}
util.save("ModelReconciliation/merge_results", merge_results)
print("\nMerge results saved to datacache.")

2026-02-03 13:42:16,890 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-03 13:42:16,890 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-03 13:42:16,891 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-02-03 13:42:16,892 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


/Users/chenry/Dropbox/Projects/KBUtilLib/src


2026-02-03 13:42:17,267 - __main__.NotebookUtil - WARNING - BLAST tools not found. Install NCBI BLAST+ to use BLAST functionality. On Ubuntu/Debian: sudo apt-get install ncbi-blast+, On MacOS: brew install blast
2026-02-03 13:42:17,268 - __main__.NotebookUtil - INFO - Notebook name: ModelReconciliation
2026-02-03 13:42:17,269 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-03 13:42:17,270 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/


/Users/chenry/.npm-global/bin/claude


2026-02-03 13:42:18,943 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: claude-code


...

Evaluating ATP production on default media...


INFO:modelseedpy.core.msmodelutl:cpd08021 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00811 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd08021 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00811 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd11632 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd08701 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd01024 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd01024 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00187 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00425 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00441 not found in model!
INFO:modelseedpy.core.msatpcorrection:max_gapfilling: 10, best_score: 0.0
INFO:modelseedpy.core.msmodelutl:Expansion time:empty:0.0504311249999887
INFO:modelseedpy.core.msmodelutl:Filtered count:0 out of 123
INFO:modelseedpy.core.msmodelutl:Expansion time:Fum.O2:0.06538104199995587
INFO:modelseedpy.core.msmodelutl:Fi

Built 4 ATP tests from 3 selected media
Running ATP safe expansion test on added reactions...


INFO:modelseedpy.core.msmodelutl:Expansion time:Fum:0.05823820800003432
INFO:modelseedpy.core.msmodelutl:Filtered count:0 out of 123



ATP expansion filter results:
  Reactions/directions tested: 123
  Filtered (ATP-breaking): 0
  Unique reactions removed or constrained: 0

Final merged model: 1504 rxns, 1343 mets, 1840 genes
  Published reactions retained: 62 / 62

Merge results saved to datacache.


## Cell 7: Run pFBA and FVA on Merged Model in Pyruvate Media

Run pFBA and FVA (50% optimum) on the merged model in pyruvate minimal media.
This enables comparison of the merged model's flux distribution against the original models.

In [8]:
%run util.py

# Load the merged model saved by Cell 6
# (Cell 6 saves merge_results to datacache; we reload the model from local JSON and re-merge)
ms_mdlutl = MSModelUtil.from_cobrapy("models/ModelSEED_ADP1.json")
ms_model = ms_mdlutl.model
pub_mdlutl = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")
pub_model = pub_mdlutl.model

# Load merge results to know which reactions to add back
merge_results = util.load("ModelReconciliation/merge_results")
retained_rxn_ids = set(merge_results["retained_rxn_ids"])
print(f"Re-adding {len(retained_rxn_ids)} retained published reactions to ModelSEED model")

# Add retained reactions from published model with normalized compartments
reactions_to_add = []
exchange_rxns_to_add = []
for rxn_id in retained_rxn_ids:
    if rxn_id in ms_model.reactions:
        continue  # Already exists
    pub_rxn = pub_model.reactions.get_by_id(rxn_id)
    new_rxn = Reaction(pub_rxn.id)
    new_rxn.name = pub_rxn.name
    new_rxn.lower_bound = pub_rxn.lower_bound
    new_rxn.upper_bound = pub_rxn.upper_bound
    new_rxn.gene_reaction_rule = pub_rxn.gene_reaction_rule
    met_dict = {}
    for met, coeff in pub_rxn.metabolites.items():
        if met.id in ms_model.metabolites:
            met_dict[ms_model.metabolites.get_by_id(met.id)] = coeff
        else:
            new_met = Metabolite(met.id, name=met.name,
                                 compartment=util.normalize_compartment(met.compartment),
                                 formula=met.formula, charge=met.charge)
            met_dict[new_met] = coeff
    new_rxn.add_metabolites(met_dict)
    reactions_to_add.append(new_rxn)

ms_model.add_reactions(reactions_to_add)

# Add exchange reactions for new extracellular metabolites
for met in ms_model.metabolites:
    if met.compartment == "e0":
        has_exchange = any(len(rxn.metabolites) == 1 and met in rxn.metabolites for rxn in met.reactions)
        if not has_exchange:
            ex_rxn = Reaction(f"EX_{met.id}")
            ex_rxn.name = f"Exchange for {met.name}"
            ex_rxn.lower_bound = -1000
            ex_rxn.upper_bound = 1000
            ex_rxn.add_metabolites({met: -1})
            exchange_rxns_to_add.append(ex_rxn)
if exchange_rxns_to_add:
    ms_model.add_reactions(exchange_rxns_to_add)

print(f"Merged model: {len(ms_model.reactions)} rxns, {len(ms_model.metabolites)} mets, {len(ms_model.genes)} genes")

# --- Run pFBA ---
print("\n=== Merged Model: pFBA in Pyruvate Media ===")
merged_pfba = util.run_fba(ms_mdlutl, media="KBaseMedia/Carbon-Pyruvic-Acid",
                            objective="MAX{bio1}", run_pfba=True)
print(f"pFBA growth rate: {merged_pfba.objective_value:.6f}")

# --- Run FVA at 50% optimum ---
merged_fva = util.run_fva(ms_mdlutl, media="KBaseMedia/Carbon-Pyruvic-Acid",
                           objective="MAX{bio1}", fraction_of_optimum=0.5)
print(f"FVA completed for {len(merged_fva)} reactions at 50% optimum")

# Package results
merged_results = {
    "growth_rate": merged_pfba.objective_value,
    "fluxes": {rxn_id: float(val) for rxn_id, val in merged_pfba.fluxes.items()},
    "fva": merged_fva
}
util.save("ModelReconciliation/merged_pyr_results", merged_results)

# Compare with original models
pub_results = util.load("ModelReconciliation/published_pyr_results")
ms_results = util.load("ModelReconciliation/modelseed_pyr_results")
print(f"\nGrowth rate comparison:")
print(f"  Published model:  {pub_results['growth_rate']:.6f}")
print(f"  ModelSEED model:  {ms_results['growth_rate']:.6f}")
print(f"  Merged model:     {merged_pfba.objective_value:.6f}")

# Classify merged FVA results
fva_classes = {}
for rxn_id in merged_fva:
    flux_val = merged_results["fluxes"].get(rxn_id, 0.0)
    fva_entry = merged_fva[rxn_id]
    fva_classes[rxn_id] = util.classify_fva_flux(fva_entry, flux_val)

# Count by class
from collections import Counter
class_counts = Counter(fva_classes.values())
print(f"\nMerged model FVA classification:")
for cls, count in sorted(class_counts.items()):
    print(f"  {cls}: {count}")

print("\nMerged model results saved to datacache.")

2026-02-03 13:42:26,722 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-03 13:42:26,722 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-03 13:42:26,723 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-02-03 13:42:26,724 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


/Users/chenry/Dropbox/Projects/KBUtilLib/src


2026-02-03 13:42:27,121 - __main__.NotebookUtil - WARNING - BLAST tools not found. Install NCBI BLAST+ to use BLAST functionality. On Ubuntu/Debian: sudo apt-get install ncbi-blast+, On MacOS: brew install blast
2026-02-03 13:42:27,123 - __main__.NotebookUtil - INFO - Notebook name: ModelReconciliation
2026-02-03 13:42:27,123 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-03 13:42:27,124 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/


/Users/chenry/.npm-global/bin/claude


2026-02-03 13:42:28,639 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: claude-code


Re-adding 62 retained published reactions to ModelSEED model
Merged model: 1504 rxns, 1343 mets, 1840 genes

=== Merged Model: pFBA in Pyruvate Media ===
pFBA growth rate: 0.209821
FVA completed for 1504 reactions at 50% optimum

Growth rate comparison:
  Published model:  0.446864
  ModelSEED model:  0.209254
  Merged model:     0.209821

Merged model FVA classification:
  blocked: 663
  essential_fwd: 228
  essential_rev: 96
  optionally_active_fwd: 35
  optionally_active_rev: 5
  optionally_active_zero_fwd: 260
  optionally_active_zero_rev: 96
  variable_fwd: 31
  variable_rev: 27
  variable_zero: 63

Merged model results saved to datacache.


## Cell 8: Save the Final Merged Model

Save the merged model (ModelSEED + retained published reactions) as a COBRApy JSON file
in `models/MergedADP1Model.json` for future use.

In [9]:
%run util.py

# Rebuild the merged model from components (for cell independence)
ms_mdlutl = MSModelUtil.from_cobrapy("models/ModelSEED_ADP1.json")
ms_model = ms_mdlutl.model
pub_mdlutl = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")
pub_model = pub_mdlutl.model

# Load merge results to know which reactions survived ATP filtering
merge_results = util.load("ModelReconciliation/merge_results")
retained_rxn_ids = set(merge_results["retained_rxn_ids"])

# Add retained reactions with normalized compartments
reactions_to_add = []
exchange_rxns_to_add = []
for rxn_id in retained_rxn_ids:
    if rxn_id in ms_model.reactions:
        continue
    pub_rxn = pub_model.reactions.get_by_id(rxn_id)
    new_rxn = Reaction(pub_rxn.id)
    new_rxn.name = pub_rxn.name
    new_rxn.lower_bound = pub_rxn.lower_bound
    new_rxn.upper_bound = pub_rxn.upper_bound
    new_rxn.gene_reaction_rule = pub_rxn.gene_reaction_rule
    met_dict = {}
    for met, coeff in pub_rxn.metabolites.items():
        if met.id in ms_model.metabolites:
            met_dict[ms_model.metabolites.get_by_id(met.id)] = coeff
        else:
            new_met = Metabolite(met.id, name=met.name,
                                 compartment=util.normalize_compartment(met.compartment),
                                 formula=met.formula, charge=met.charge)
            met_dict[new_met] = coeff
    new_rxn.add_metabolites(met_dict)
    reactions_to_add.append(new_rxn)

ms_model.add_reactions(reactions_to_add)

# Add exchange reactions for new extracellular metabolites
for met in ms_model.metabolites:
    if met.compartment == "e0":
        has_exchange = any(len(rxn.metabolites) == 1 and met in rxn.metabolites for rxn in met.reactions)
        if not has_exchange:
            ex_rxn = Reaction(f"EX_{met.id}")
            ex_rxn.name = f"Exchange for {met.name}"
            ex_rxn.lower_bound = -1000
            ex_rxn.upper_bound = 1000
            ex_rxn.add_metabolites({met: -1})
            exchange_rxns_to_add.append(ex_rxn)
if exchange_rxns_to_add:
    ms_model.add_reactions(exchange_rxns_to_add)

# Save the merged model
output_path = "models/MergedADP1Model.json"
cobra.io.save_json_model(ms_model, output_path)
print(f"Merged model saved to {output_path}")
print(f"  Reactions: {len(ms_model.reactions)}")
print(f"  Metabolites: {len(ms_model.metabolites)}")
print(f"  Genes: {len(ms_model.genes)}")
print(f"  Published reactions added: {len(retained_rxn_ids)}")

2026-02-03 13:42:49,442 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-03 13:42:49,442 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-03 13:42:49,443 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-02-03 13:42:49,444 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


/Users/chenry/Dropbox/Projects/KBUtilLib/src


2026-02-03 13:42:49,829 - __main__.NotebookUtil - WARNING - BLAST tools not found. Install NCBI BLAST+ to use BLAST functionality. On Ubuntu/Debian: sudo apt-get install ncbi-blast+, On MacOS: brew install blast
2026-02-03 13:42:49,830 - __main__.NotebookUtil - INFO - Notebook name: ModelReconciliation
2026-02-03 13:42:49,831 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-03 13:42:49,832 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/


/Users/chenry/.npm-global/bin/claude


2026-02-03 13:42:51,450 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: claude-code


Merged model saved to models/MergedADP1Model.json
  Reactions: 1504
  Metabolites: 1343
  Genes: 1840
  Published reactions added: 62


## Cell 9: Escher Map of Merged Model with Reaction Class Badges

Generate an interactive Escher map of the merged model painted with pyruvate pFBA fluxes.
Each reaction is badged with one of 6 categories combining:
- **Source**: Original ModelSEED model vs. added from published model
- **FVA class**: Essential (must carry flux), Variable (flux range spans zero), or Blocked (no flux)

Badge categories:
1. Essential + Original ModelSEED
2. Variable + Original ModelSEED
3. Blocked + Original ModelSEED
4. Essential + Added from Published
5. Variable + Added from Published
6. Blocked + Added from Published

Uses `util.create_map_html2()` from KBUtilLib's `EscherUtils` with the `reaction_classes` parameter.

In [10]:
%run util.py

# Load the merged model
merged_mdlutl = MSModelUtil.from_cobrapy("models/MergedADP1Model.json")
merged_model = merged_mdlutl.model

# Load merge results and flux/FVA results
merge_results = util.load("ModelReconciliation/merge_results")
merged_results = util.load("ModelReconciliation/merged_pyr_results")
retained_rxn_ids = set(merge_results["retained_rxn_ids"])

fluxes = merged_results["fluxes"]
fva = merged_results["fva"]

# Build the set of original ModelSEED reaction IDs (everything NOT in retained_rxn_ids)
original_ms_rxn_ids = set(rxn.id for rxn in merged_model.reactions) - retained_rxn_ids

# Classify each reaction into one of the 6 badge categories
# FVA classification: essential (fwd or rev), variable (any variable_*), blocked
def simplify_fva_class(fva_class):
    """Map detailed FVA class to simple: essential, variable, or blocked."""
    if "essential" in fva_class:
        return "Essential"
    elif "blocked" in fva_class:
        return "Blocked"
    else:
        return "Variable"  # optionally_active, variable_fwd/rev/zero, etc.

reaction_classes = {}
for rxn in merged_model.reactions:
    # Skip exchange reactions and biomass from badge assignment
    if len(rxn.metabolites) == 1:
        continue
    if rxn.id.startswith("bio") or "GROWTH" in rxn.id:
        continue
    
    # Determine source
    if rxn.id in retained_rxn_ids:
        source = "Published"
    else:
        source = "ModelSEED"
    
    # Determine FVA class
    flux_val = fluxes.get(rxn.id, 0.0)
    fva_entry = fva.get(rxn.id, {"MIN": 0, "MAX": 0})
    detailed_class = util.classify_fva_flux(fva_entry, flux_val)
    simple_class = simplify_fva_class(detailed_class)
    
    # Combine into badge category
    reaction_classes[rxn.id] = f"{simple_class} ({source})"

# Print badge category counts
from collections import Counter
badge_counts = Counter(reaction_classes.values())
print("Reaction badge categories:")
for badge, count in sorted(badge_counts.items()):
    print(f"  {badge}: {count}")
print(f"  Total badged: {len(reaction_classes)}")

# Generate Escher map
output_path = f"{util.output_dir}/ModelReconciliation/merged_model_escher.html"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# List available maps to find the best one for this model
print("\nSearching for best Escher map...")
map_df = util.list_available_maps(model=merged_mdlutl, as_df=True)
if len(map_df) > 0:
    map_df_sorted = map_df.sort_values("model_reaction_coverage", ascending=False)
    print(f"Top maps by coverage:")
    for _, row in map_df_sorted.head(5).iterrows():
        print(f"  {row['name']} ({row['source']}): {row.get('model_reaction_coverage', 0):.1%} coverage, "
              f"{row.get('model_reactions_in_map', 0)} reactions")
    best_map = map_df_sorted.iloc[0]["name"]
else:
    best_map = "core"

print(f"\nUsing map: {best_map}")

# Create the Escher map with flux data and reaction class badges
util.create_map_html2(
    model=merged_mdlutl,
    map=best_map,
    output_path=output_path,
    flux=fluxes,
    reaction_classes=reaction_classes,
    height=800,
    width=1200
)

print(f"\nEscher map saved to: {output_path}")
print(f"Open in browser to view the interactive map with reaction class badges.")

# Save badge assignments to datacache
util.save("ModelReconciliation/reaction_badges", reaction_classes)

2026-02-03 13:42:58,642 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-03 13:42:58,643 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-03 13:42:58,643 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-02-03 13:42:58,644 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


/Users/chenry/Dropbox/Projects/KBUtilLib/src


2026-02-03 13:42:59,040 - __main__.NotebookUtil - WARNING - BLAST tools not found. Install NCBI BLAST+ to use BLAST functionality. On Ubuntu/Debian: sudo apt-get install ncbi-blast+, On MacOS: brew install blast
2026-02-03 13:42:59,041 - __main__.NotebookUtil - INFO - Notebook name: ModelReconciliation
2026-02-03 13:42:59,042 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-03 13:42:59,042 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/


/Users/chenry/.npm-global/bin/claude


2026-02-03 13:43:00,550 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: claude-code


Reaction badge categories:
  Blocked (ModelSEED): 539
  Blocked (Published): 41
  Essential (ModelSEED): 304
  Variable (ModelSEED): 435
  Variable (Published): 21
  Total badged: 1340

Searching for best Escher map...


2026-02-03 13:43:02,301 - __main__.NotebookUtil - WARNING - Map data does not have expected Escher or KBase structure
2026-02-03 13:43:02,724 - __main__.NotebookUtil - INFO - Loaded map 'full' from local index: /Users/chenry/Dropbox/Projects/KBUtilLib/src/kbutillib/../../data/escher_maps/full.json
2026-02-03 13:43:02,743 - __main__.NotebookUtil - INFO - Updated names for 759 reactions in map


Top maps by coverage:
  full (local): 34.6% coverage, 520 reactions
  core (local): 8.3% coverage, 125 reactions
  Narrative.1624896051323 (kbase): 0.0% coverage, 0 reactions
  o0u5z7uvvADq (kbase): 0.0% coverage, 0 reactions

Using map: full


2026-02-03 13:43:03,732 - __main__.NotebookUtil - INFO - Injected reaction class overlays for 1340 reactions across 5 classes



Escher map saved to: /Users/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/nboutput/ModelReconciliation/ModelReconciliation/merged_model_escher.html
Open in browser to view the interactive map with reaction class badges.
